# Duelo Final — Random Forest Avançado
### Competição Titanic | Kaggle

**Estratégia:** Superar os resultados da aula utilizando Random Forest com:
- Feature engineering avançado (novas variáveis)
- Mais features do que as usadas em aula
- RandomizedSearchCV para otimização de hiperparâmetros
- StratifiedKFold para validação cruzada mais robusta
- SMOTE + StandardScaler no pré-processamento

## 1. Importação das Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Modelo principal
from sklearn.ensemble import RandomForestClassifier

# Otimização de hiperparâmetros
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc           # necessário para calcular a área sob a curva ROC
)

# Reprodutibilidade
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

print('Bibliotecas importadas com sucesso!')

## 2. Carregamento dos Dados

In [ ]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print(f'Treino: {train_df.shape[0]} linhas x {train_df.shape[1]} colunas')
print(f'Teste:  {test_df.shape[0]} linhas x {test_df.shape[1]} colunas')

In [ ]:
train_df.head()

In [ ]:
# Visão geral dos dados de treino
train_df.info()

In [ ]:
# Verificando valores nulos
print('Valores nulos — Treino:')
print(train_df.isnull().sum())
print()
print('Valores nulos — Teste:')
print(test_df.isnull().sum())

In [ ]:
# Distribuição da variável alvo
print('Distribuição da variável alvo (Survived):')
print(train_df['Survived'].value_counts())
print()
print(f'Taxa de sobrevivência: {train_df["Survived"].mean():.2%}')

## 3. Análise Exploratória dos Dados (EDA)

Antes de qualquer modelagem, precisamos entender os dados.
A EDA nos ajuda a identificar padrões, relações entre variáveis e
decisões de pré-processamento — como quais features incluir e como
tratar valores nulos.

Perguntas que queremos responder:
- Quais variáveis têm maior relação com a sobrevivência?
- Como se distribuem idade, tarifa e classe entre sobreviventes e não sobreviventes?
- Existe correlação forte entre as variáveis numéricas?

In [ ]:
# Taxas de sobrevivência pelas principais variáveis categóricas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Taxa de sobrevivência por variável categórica', fontsize=14, y=1.02)

# Por Sexo
survival_sex = train_df.groupby('Sex')['Survived'].mean()
survival_sex.index = ['Masculino', 'Feminino']
survival_sex.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_title('Por Sexo')
axes[0].set_ylabel('Taxa de sobrevivência')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.0%}',
                     (p.get_x() + p.get_width() / 2, p.get_height() + 0.02),
                     ha='center')

# Por Classe
survival_pclass = train_df.groupby('Pclass')['Survived'].mean()
survival_pclass.plot(kind='bar', ax=axes[1], color=['gold', 'silver', '#cd7f32'], edgecolor='white')
axes[1].set_title('Por Classe do Bilhete')
axes[1].set_ylabel('Taxa de sobrevivência')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.0%}',
                     (p.get_x() + p.get_width() / 2, p.get_height() + 0.02),
                     ha='center')

# Por Porto de Embarque
survival_embarked = train_df.groupby('Embarked')['Survived'].mean()
survival_embarked.plot(kind='bar', ax=axes[2], color='teal', edgecolor='white')
axes[2].set_title('Por Porto de Embarque')
axes[2].set_ylabel('Taxa de sobrevivência')
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=0)
for p in axes[2].patches:
    axes[2].annotate(f'{p.get_height():.0%}',
                     (p.get_x() + p.get_width() / 2, p.get_height() + 0.02),
                     ha='center')

plt.tight_layout()
plt.show()

print('Observação: mulheres tiveram taxa de sobrevivência muito superior (74% vs 19%).')
print('Passageiros de 1ª classe também sobreviveram em maior proporção (63% vs 24% na 3ª).')

In [ ]:
# Distribuição de Idade e Tarifa entre sobreviventes e não sobreviventes
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Distribuição de variáveis numéricas por sobrevivência', fontsize=14)

# Idade
for survived, label, color in [(0, 'Não sobreviveu', 'steelblue'), (1, 'Sobreviveu', 'coral')]:
    subset = train_df[train_df['Survived'] == survived]['Age'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_title('Distribuição de Idade')
axes[0].set_xlabel('Idade')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Tarifa (limitando para melhor visualização — removendo outliers extremos)
for survived, label, color in [(0, 'Não sobreviveu', 'steelblue'), (1, 'Sobreviveu', 'coral')]:
    subset = train_df[(train_df['Survived'] == survived) & (train_df['Fare'] < 200)]['Fare']
    axes[1].hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
axes[1].set_title('Distribuição de Tarifa (até £200)')
axes[1].set_xlabel('Tarifa (£)')
axes[1].set_ylabel('Frequência')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Observação: crianças (Age < 10) tiveram maior taxa de sobrevivência.')
print('Tarifas mais altas (1ª classe) estão associadas a maior sobrevivência.')

In [ ]:
# Heatmap de correlação entre variáveis numéricas
# Isso nos ajuda a identificar features redundantes e relações com a variável alvo

colunas_numericas = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr = train_df[colunas_numericas].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Heatmap de Correlação — Variáveis Numéricas')
plt.tight_layout()
plt.show()

print('Principais correlações com Survived:')
print(corr['Survived'].drop('Survived').sort_values(key=abs, ascending=False).round(3))
print()
print('Observação: Pclass tem correlação negativa forte (-0.34):')
print('quanto maior a classe (3ª), menor a chance de sobrevivência.')
print('Fare tem correlação positiva (0.26): tarifas mais altas = mais sobreviventes.')

## 4. Feature Engineering

Esta é a principal diferença em relação à aula. Enquanto a aula utilizou apenas 4 features
(`Sex`, `Cabin_known`, `Fare`, `Pclass`), vamos criar novas variáveis que capturam
informações mais ricas sobre cada passageiro.

A EDA que fizemos acima orientou diretamente estas escolhas:
- A diferença de sobrevivência por **sexo** e **classe** confirmou que essas são features essenciais
- A análise de títulos revelou padrões sociais que vão além do sexo isolado
- A relação entre tamanho de família e sobrevivência justifica criar `FamilySize` e `IsAlone`

**Novas features criadas:**
- `Title`: título social extraído do nome (Mr, Mrs, Miss, Master, Rare)
- `FamilySize`: tamanho total da família a bordo
- `IsAlone`: indicador se o passageiro estava sozinho
- `FarePerPerson`: tarifa dividida pelo tamanho da família
- `Cabin_known`: indica se a cabine é conhecida

In [ ]:
# Antes de definir quais títulos são 'raros', precisamos entender
# quais títulos existem nos dados e quantas vezes aparecem.
# Essa análise justifica as escolhas feitas na função criar_features().

# Regex utilizado: r',\s(?:the\s+)?([A-Za-z]+)\.'
# O trecho (?:the\s+)? é um grupo não-capturante opcional que ignora
# o artigo 'the' presente em nomes como 'Rothes, the Countess.'
# Sem isso, o regex capturaria 'the' em vez de 'Countess',
# gerando um título desconhecido que seria mapeado como 0.

regex_titulo = r',\s(?:the\s+)?([A-Za-z]+)\.'
# Extraindo todos os títulos presentes no treino e no teste
titulos_treino = train_df['Name'].str.extract(r',\s([A-Za-z]+)\.')[0].value_counts()
titulos_teste  = test_df['Name'].str.extract(r',\s([A-Za-z]+)\.')[0].value_counts()

print('=== Títulos encontrados no TREINO ===')
print(titulos_treino)
print()
print('=== Títulos encontrados no TESTE ===')
print(titulos_teste)

In [ ]:
# Analisando a taxa de sobrevivência por título no treino
# Isso nos ajuda a decidir quais títulos têm volume suficiente
# para serem mantidos como categoria própria e quais devem
# ser agrupados em 'Rare' por terem poucos registros.

train_df['Title_temp'] = train_df['Name'].str.extract(r',\s([A-Za-z]+)\.')

analise_titulos = (
    train_df.groupby('Title_temp')['Survived']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'taxa_sobrevivencia', 'count': 'total'})
    .sort_values('total', ascending=False)
)
analise_titulos['taxa_sobrevivencia'] = analise_titulos['taxa_sobrevivencia'].round(2)

print('=== Taxa de sobrevivência por título ===')
print(analise_titulos)
print()
print('Critério adotado: títulos com menos de 10 ocorrências no treino')
print('serão agrupados como Rare para evitar ruído no modelo.')
print()
raros = analise_titulos[analise_titulos['total'] < 10].index.tolist()
print(f'Títulos classificados como Rare: {raros}')

# Removendo coluna temporária
train_df.drop(columns=['Title_temp'], inplace=True)

In [ ]:
def criar_features(df):
    """
    Aplica todas as transformações de feature engineering.
    Recebe um DataFrame e retorna o mesmo com novas colunas.
    """
    df = df.copy()

    # --- Extração do Título a partir do Nome ---
    # O nome tem o formato: 'Sobrenome, Titulo. Nome'
    df['Title'] = df['Name'].str.extract(r',\s([A-Za-z]+)\.')

    # Títulos raros: definidos com base na análise exploratória acima.
    # Critério: títulos com menos de 10 ocorrências no treino foram
    # agrupados em 'Rare' para evitar que o modelo aprenda padrões
    # de categorias com poucos exemplos (o que causaria ruído).
    # Títulos mantidos como categoria própria: Mr, Miss, Mrs, Master
    # (todos com 40+ ocorrências e taxas de sobrevivência distintas).
    # Dona foi adicionada pois aparece no teste mas não no treino.
    titulos_raros = ['Dona', 'Lady', 'Countess', 'Capt', 'Col', 'Don',
                     'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Mlle', 'Ms', 'Mme']
    df['Title'] = df['Title'].replace(titulos_raros, 'Rare')

    # Codificando o título como número
    mapa_titulos = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Rare': 5}
    df['Title'] = df['Title'].map(mapa_titulos).fillna(0).astype(int)

    # --- Tamanho da Família ---
    # SibSp = irmãos + cônjuge | Parch = pais + filhos
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 = o próprio passageiro

    # --- Indicador de viajante solo ---
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # --- Tarifa por pessoa ---
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']

    # --- Cabin: conhecido ou não ---
    df['Cabin_known'] = df['Cabin'].notna().astype(int)

    return df


train_df = criar_features(train_df)
test_df  = criar_features(test_df)

print('Feature engineering aplicado com sucesso!')
print()
print('Distribuição dos títulos após agrupamento:')
print(train_df['Title'].value_counts())
print('(1=Mr, 2=Miss, 3=Mrs, 4=Master, 5=Rare)')

In [ ]:
# Visualizando a taxa de sobrevivência por título
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Por Título
title_survival = train_df.groupby('Title')['Survived'].mean()
title_survival.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Taxa de sobrevivência por Título')
axes[0].set_xlabel('Título (1=Mr, 2=Miss, 3=Mrs, 4=Master, 5=Rare)')
axes[0].set_ylabel('Taxa de sobrevivência')
axes[0].tick_params(axis='x', rotation=0)

# Por tamanho de família
family_survival = train_df.groupby('FamilySize')['Survived'].mean()
family_survival.plot(kind='bar', ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Taxa de sobrevivência por Tamanho de Família')
axes[1].set_xlabel('FamilySize')
axes[1].set_ylabel('Taxa de sobrevivência')
axes[1].tick_params(axis='x', rotation=0)

# IsAlone
alone_survival = train_df.groupby('IsAlone')['Survived'].mean()
alone_survival.plot(kind='bar', ax=axes[2], color='coral', edgecolor='white')
axes[2].set_title('Taxa de sobrevivência: Solo vs Acompanhado')
axes[2].set_xlabel('IsAlone (0=acompanhado, 1=solo)')
axes[2].set_ylabel('Taxa de sobrevivência')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 5. Pré-processamento dos Dados

Com a EDA e o feature engineering feitos, precisamos deixar os dados
no formato que o modelo consegue processar. Isso envolve três etapas:

**Codificação de variáveis categóricas:**
Modelos de machine learning trabalham com números, não com texto.
Usamos estratégias diferentes dependendo da variável:
- `Sex`: apenas dois valores (male/female) → **Label Encoding** (0 e 1)
- `Embarked`: três valores (C, Q, S) → **One-Hot Encoding** para evitar
  que o modelo interprete uma ordenação inexistente entre os portos

**Imputação de valores nulos:**
Optamos pela **mediana por grupo** (Pclass + Sex) em vez da média global.
O raciocínio é simples: a distribuição etária dos passageiros de 1ª classe
era muito diferente da 3ª classe — usar uma média global mascararia essa
diferença e introduziria ruído no modelo.

**Seleção de features:**
Utilizamos 14 variáveis — mais que o triplo das 4 usadas na aula —
incluindo todas as novas features criadas na etapa anterior.

In [ ]:
# Codificação de variáveis categóricas

# Sex: Label Encoding (apenas 2 categorias)
train_df['Sex'] = train_df['Sex'].map({'male': 0, 'female': 1})
test_df['Sex']  = test_df['Sex'].map({'male': 0, 'female': 1})

# Embarked: One-Hot Encoding (mais de 2 categorias)
train_df = pd.get_dummies(train_df, columns=['Embarked'], prefix='Embarked')
test_df  = pd.get_dummies(test_df,  columns=['Embarked'], prefix='Embarked')

# Garantindo que as colunas Embarked existam em ambos os datasets
for col in ['Embarked_C', 'Embarked_Q', 'Embarked_S']:
    if col not in test_df.columns:
        test_df[col] = 0

print('Codificação aplicada com sucesso!')

In [ ]:
# Tratamento de valores nulos

# Age: imputamos com a mediana por grupo (Pclass + Sex)
# Mediana é mais robusta que a média quando há outliers
for df in [train_df, test_df]:
    df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
        lambda x: x.fillna(x.median())
    )

# Fare: apenas 1 nulo no teste — usamos a mediana por Pclass
test_df['Fare'] = test_df.groupby('Pclass')['Fare'].transform(
    lambda x: x.fillna(x.median())
)

# Recalculando FarePerPerson após imputação
train_df['FarePerPerson'] = train_df['Fare'] / train_df['FamilySize']
test_df['FarePerPerson']  = test_df['Fare']  / test_df['FamilySize']

print('Valores nulos tratados!')
print()
print('Nulos restantes — Treino:', train_df.isnull().sum().sum())
print('Nulos restantes — Teste: ', test_df.isnull().sum().sum())

In [ ]:
# Definindo as features que serão usadas no modelo
# Aqui usamos MUITO mais variáveis do que na aula (que usou apenas 4)

FEATURES = [
    'Pclass',         # Classe do bilhete
    'Sex',            # Gênero
    'Age',            # Idade (com imputação inteligente)
    'SibSp',          # Irmãos/cônjuge
    'Parch',          # Pais/filhos
    'Fare',           # Tarifa
    'Cabin_known',    # Cabine conhecida?
    'Title',          # Título social (NOVO)
    'FamilySize',     # Tamanho da família (NOVO)
    'IsAlone',        # Viajando sozinho? (NOVO)
    'FarePerPerson',  # Tarifa por pessoa (NOVO)
    'Embarked_C',     # Porto de embarque
    'Embarked_Q',
    'Embarked_S',
]

X      = train_df[FEATURES]
y      = train_df['Survived']
X_test = test_df[FEATURES]

print(f'Features utilizadas: {len(FEATURES)}')
print(f'Shape X treino: {X.shape}')
print(f'Shape X teste:  {X_test.shape}')

## 6. Balanceamento e Padronização

**Por que balancear?**
O conjunto de treino tem 549 não sobreviventes (62%) contra 342 sobreviventes (38%).
Esse desbalanceamento faz com que o modelo tenda a prever sempre a classe majoritária,
pois isso já garante uma boa acurácia. Para corrigir isso, usamos o **SMOTE**
(Synthetic Minority Over-sampling Technique), que gera amostras sintéticas da
classe minoritária — ao contrário do undersampling, que descartaria dados reais.

**Por que padronizar?**
Embora o Random Forest seja menos sensível à escala do que o SVM, a padronização
ainda contribui para a estabilidade e convergência do modelo, especialmente quando
as features têm magnitudes muito diferentes — por exemplo, `Fare` pode chegar a 512
enquanto `IsAlone` é sempre 0 ou 1.

**Ponto importante:** o `StandardScaler` é ajustado (`fit`) apenas nos dados de treino
e apenas aplicado (`transform`) nos dados de teste. Fazer `fit` no teste seria
**data leakage** — o modelo estaria 'vendo' informações do teste antes da hora.

In [ ]:
# Verificando o desbalanceamento antes do SMOTE
print('Distribuição antes do balanceamento:')
print(y.value_counts())
print(f'Proporção: {y.mean():.2%} de sobreviventes')

In [ ]:
# Aplicando SMOTE para balancear as classes
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

print('Distribuição após SMOTE:')
print(pd.Series(y_balanced).value_counts())

In [ ]:
# Padronizando os dados (importante para estabilidade do modelo)
scaler = StandardScaler()
X_balanced_scaled = scaler.fit_transform(X_balanced)
X_test_scaled     = scaler.transform(X_test)

print('Padronização aplicada!')
print(f'Shape treino balanceado e padronizado: {X_balanced_scaled.shape}')

## 7. Duelo 1 — Modelo Base (sem otimização)

Começamos com o Random Forest usando apenas os parâmetros padrão.
Esse passo é fundamental: sem um baseline, não temos como saber
se a otimização de hiperparâmetros realmente trouxe ganho real.
Todo experimento sério em ciência de dados começa medindo o ponto de partida.

In [ ]:
# Modelo base — sem ajuste de hiperparâmetros
rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_balanced_scaled, y_balanced)

y_pred_base = rf_base.predict(X_balanced_scaled)

print('=== Random Forest — Modelo Base ===')
print(classification_report(y_balanced, y_pred_base))

In [ ]:
# Cross-validation com StratifiedKFold no modelo base
# StratifiedKFold garante que cada fold tenha a mesma proporção de classes
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_base = cross_val_score(
    rf_base, X_balanced_scaled, y_balanced,
    cv=skf, scoring='accuracy'
)

print('Cross-Validation — Modelo Base:')
print(f'Scores por fold: {cv_scores_base.round(4)}')
print(f'Média:  {cv_scores_base.mean():.4f}')
print(f'Desvio: {cv_scores_base.std():.4f}')

## 8. Duelo 2 — Otimização com RandomizedSearchCV

Utilizamos `RandomizedSearchCV` em vez do `GridSearchCV` da aula.

**Por que RandomizedSearchCV?**
O espaço de hiperparâmetros que definimos tem mais de **3.000 combinações possíveis**.
Testar todas com GridSearchCV e 5 folds seria computacionalmente inviável.
O RandomizedSearchCV testa 100 combinações escolhidas aleatoriamente — e estatisticamente
tende a encontrar resultados tão bons quanto o GridSearch em uma fração do tempo.

**Por que StratifiedKFold?**
O KFold simples sorteia os folds aleatoriamente, o que pode gerar folds com proporções
de classes diferentes. O StratifiedKFold garante que cada fold tenha a mesma proporção
de sobreviventes e não sobreviventes — avaliação mais justa e estável.

In [ ]:
# Grade de hiperparâmetros — muito mais ampla do que na aula
param_dist = {
    'n_estimators':      [100, 200, 300, 500],        # Número de árvores
    'max_depth':         [None, 5, 10, 15, 20, 30],   # Profundidade máxima
    'min_samples_split': [2, 5, 10, 15],              # Mín. amostras para dividir nó
    'min_samples_leaf':  [1, 2, 4, 6],                # Mín. amostras por folha
    'max_features':      ['sqrt', 'log2', None],      # Features por divisão
    'bootstrap':         [True, False],               # Amostragem com reposição
    'class_weight':      ['balanced', None],          # Pesos das classes
}

print('Grade de hiperparâmetros definida!')
total = 4 * 6 * 4 * 4 * 3 * 2 * 2
print(f'Total de combinações possíveis: {total:,}')
print('Número de combinações testadas pelo RandomizedSearchCV: 100')

In [ ]:
# Executando o RandomizedSearchCV
rf_otimizado = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf_otimizado,
    param_distributions=param_dist,
    n_iter=100,               # Número de combinações a testar
    scoring='accuracy',
    cv=skf,                   # StratifiedKFold com 5 folds
    n_jobs=-1,                # Usa todos os processadores disponíveis
    random_state=42,
    verbose=1
)

random_search.fit(X_balanced_scaled, y_balanced)

print()
print('=== Resultados do RandomizedSearchCV ===')
print(f'Melhores parâmetros: {random_search.best_params_}')
print(f'Melhor acurácia (CV): {random_search.best_score_:.4f}')

In [ ]:
# Avaliando o melhor modelo no conjunto de treino
melhor_modelo = random_search.best_estimator_
y_pred_otimizado = melhor_modelo.predict(X_balanced_scaled)

print('=== Random Forest — Modelo Otimizado ===')
print(classification_report(y_balanced, y_pred_otimizado))

In [ ]:
# Cross-validation do modelo otimizado
cv_scores_otimizado = cross_val_score(
    melhor_modelo, X_balanced_scaled, y_balanced,
    cv=skf, scoring='accuracy'
)

print('Cross-Validation — Modelo Otimizado:')
print(f'Scores por fold: {cv_scores_otimizado.round(4)}')
print(f'Média:  {cv_scores_otimizado.mean():.4f}')
print(f'Desvio: {cv_scores_otimizado.std():.4f}')
print()
print(f'Melhora em relação ao modelo base: '
      f'{(cv_scores_otimizado.mean() - cv_scores_base.mean()):.4f}')

## 9. Análise de Importância das Features

Uma das grandes vantagens do Random Forest é a interpretabilidade: ele nos diz
automaticamente quais features contribuíram mais para as decisões do modelo.
Isso é especialmente útil para validar o feature engineering — se as features
novas que criamos (`Title`, `FamilySize`, `IsAlone`) aparecerem com alta importância,
significa que elas realmente agregaram valor ao modelo.

In [ ]:
# Importância das features
importancias = pd.Series(
    melhor_modelo.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=importancias.values,
    y=importancias.index,
    palette='Blues_r'
)
plt.title('Importância das Features — Random Forest Otimizado')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

print('\nImportância por feature:')
print(importancias.round(4))

## 10. Matriz de Confusão

A matriz de confusão mostra em detalhe onde o modelo erra:
- **Verdadeiro Positivo (VP):** previu sobreviveu, realmente sobreviveu
- **Verdadeiro Negativo (VN):** previu não sobreviveu, realmente não sobreviveu
- **Falso Positivo (FP):** previu sobreviveu, mas não sobreviveu (erro tipo I)
- **Falso Negativo (FN):** previu não sobreviveu, mas sobreviveu (erro tipo II)

In [ ]:
# Matriz de confusão do modelo otimizado
cm = confusion_matrix(y_balanced, y_pred_otimizado)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Não sobreviveu', 'Sobreviveu'],
    yticklabels=['Não sobreviveu', 'Sobreviveu']
)
plt.title('Matriz de Confusão — Random Forest Otimizado')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.tight_layout()
plt.show()

## 11. Curva ROC e AUC

A **Curva ROC** (Receiver Operating Characteristic) é uma das métricas mais
usadas em competições do Kaggle. Ela mostra o trade-off entre a taxa de
verdadeiros positivos (sensibilidade) e falsos positivos em diferentes limiares de decisão.

O **AUC** (Area Under the Curve) resume a curva em um único número:
- AUC = 1.0 → modelo perfeito
- AUC = 0.5 → modelo aleatório (sem valor preditivo)
- AUC > 0.8 → modelo considerado bom na prática

In [ ]:
# Calculando a curva ROC para o modelo base e o otimizado
# Usamos predict_proba para obter as probabilidades (não apenas 0/1)

y_prob_base      = rf_base.predict_proba(X_balanced_scaled)[:, 1]
y_prob_otimizado = melhor_modelo.predict_proba(X_balanced_scaled)[:, 1]

fpr_base, tpr_base, _ = roc_curve(y_balanced, y_prob_base)
fpr_otim, tpr_otim, _ = roc_curve(y_balanced, y_prob_otimizado)

auc_base = roc_auc_score(y_balanced, y_prob_base)
auc_otim = roc_auc_score(y_balanced, y_prob_otimizado)

# Plotando
plt.figure(figsize=(8, 6))
plt.plot(fpr_base, tpr_base,
         color='steelblue', lw=2,
         label=f'Modelo Base (AUC = {auc_base:.3f})')
plt.plot(fpr_otim, tpr_otim,
         color='coral', lw=2,
         label=f'Modelo Otimizado (AUC = {auc_otim:.3f})')
plt.plot([0, 1], [0, 1],
         color='gray', lw=1, linestyle='--',
         label='Modelo aleatório (AUC = 0.500)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos (Sensibilidade)')
plt.title('Curva ROC — Modelo Base vs Otimizado')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC Modelo Base:      {auc_base:.4f}')
print(f'AUC Modelo Otimizado: {auc_otim:.4f}')
print(f'Ganho de AUC:         +{(auc_otim - auc_base):.4f}')

## 12. Comparativo: Base vs Otimizado

Chegou a hora de responder a pergunta mais importante do experimento:
**a otimização de hiperparâmetros realmente fez diferença?**

O comparativo abaixo mostra a acurácia média no cross-validation (5 folds)
de cada modelo. Usar a média do CV em vez da acurácia no treino é fundamental —
a acurácia no treino sempre será alta (o modelo 'viu' esses dados), enquanto
o CV estima como o modelo vai se comportar em dados novos, como o conjunto
de teste do Kaggle.

O desvio padrão entre os folds também importa: um desvio alto indica que
o modelo é instável e pode ter desempenho muito diferente dependendo dos dados.

In [ ]:
# Resumo comparativo
print('=' * 50)
print('COMPARATIVO DE MODELOS')
print('=' * 50)
print(f'Modelo Base     — CV Accuracy: {cv_scores_base.mean():.4f} (± {cv_scores_base.std():.4f})')
print(f'Modelo Otimizado— CV Accuracy: {cv_scores_otimizado.mean():.4f} (± {cv_scores_otimizado.std():.4f})')
print()
print(f'Melhora obtida: +{(cv_scores_otimizado.mean() - cv_scores_base.mean()) * 100:.2f} pontos percentuais')
print('=' * 50)

## 13. Geração das Previsões para o Kaggle

Utilizamos o melhor modelo encontrado pelo RandomizedSearchCV para gerar
as previsões finais sobre o conjunto de teste.
O arquivo gerado segue o formato exigido pela competição do Kaggle:
duas colunas — `PassengerId` e `Survived`.

In [ ]:
# Gerando previsões com o melhor modelo
y_pred_kaggle = melhor_modelo.predict(X_test_scaled)

print(f'Previsões geradas: {len(y_pred_kaggle)}')
print(f'Distribuição das previsões:')
print(pd.Series(y_pred_kaggle).value_counts())
print(f'Taxa prevista de sobrevivência: {y_pred_kaggle.mean():.2%}')

In [ ]:
# Criando o arquivo de submissão
submissao = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived':    y_pred_kaggle
})

submissao.to_csv('submission_random_forest.csv', index=False)

print('Arquivo de submissão gerado: submission_random_forest.csv')
print()
print('Prévia das primeiras linhas:')
print(submissao.head(10))

## 14. Conclusão e Análise Crítica

### O que foi feito de diferente em relação à aula

| Aspecto | Aula (M41) | Este notebook |
|---|---|---|
| Modelo | SVM / XGBoost | Random Forest |
| Análise exploratória | Não realizada | EDA completa com 5 gráficos |
| Nº de features | 4 | 14 |
| Feature engineering | Nenhum | Title, FamilySize, IsAlone, FarePerPerson |
| Imputação de Age | Média global | Mediana por Pclass + Sex |
| Otimização | GridSearchCV | RandomizedSearchCV (100 iter.) |
| Cross-validation | KFold padrão | StratifiedKFold (5 folds) |
| Métricas apresentadas | Accuracy + classification report | + Curva ROC e AUC |

---

### O que funcionou bem

O **feature engineering** foi o maior diferencial. A criação do `Title` capturou
relações sociais que vão além do sexo — por exemplo, `Master` (crianças do sexo
masculino) teve taxa de sobrevivência muito superior ao `Mr` (adultos masculinos),
o que o modelo não conseguiria capturar usando apenas a variável `Sex`.

A **imputação de Age por grupo** (mediana por Pclass + Sex) foi mais inteligente
do que a média global, pois respeita o fato de que a distribuição etária era
diferente entre classes sociais.

O **RandomizedSearchCV** explorou um espaço muito maior de hiperparâmetros
do que o GridSearchCV da aula, sem o custo computacional de testar todas as
combinações.

---

### O que poderia ser melhorado em um próximo ciclo

- **Ensemble de modelos:** combinar Random Forest com XGBoost via VotingClassifier
  ou Stacking tende a superar qualquer modelo isolado
- **Mais feature engineering:** criar faixas etárias (criança/adulto/idoso),
  categoria de tarifa (baixa/média/alta) e prefixo da cabine como variável
- **Optuna no lugar de RandomizedSearchCV:** framework moderno de otimização
  bayesiana que aprende quais regiões do espaço de hiperparâmetros explorar

---

### Arquivo para submissão no Kaggle

`submission_random_forest.csv`

Submeta em: https://www.kaggle.com/competitions/titanic